# 04 — Feature Selection

**What this notebook is for.** Our tuned model plateaued (~0.803 AUROC, very
stable: seed std ~0.0011). Feature selection here has two goals: mainly a
**smaller, simpler, more interpretable model at the same AUROC**, and possibly a
small **improvement** if removing noisy features sharpens the model.

**Honest expectation.** Trees already ignore useless columns, so cutting features
rarely *raises* AUROC much — the likely win is "same score, far fewer features."
An improvement is possible (most likely from dropping the ~90 sparse ICD flags),
but small.

**The decision rule (from the seed-stability run).** Baseline = **0.8032**,
seed std = **0.0011**. So:
- a cut **holds** if reduced AUROC ≥ 0.8032 − 0.002 ≈ **0.8012** (within noise);
- a cut is a real **improvement** only if AUROC ≥ 0.8032 + 0.0022 ≈ **0.8054**.

**Method — tiered ablation (careful with run budget).** Instead of testing every
feature, we ablate in *meaningful groups*, each answering one question. ~8 CV
runs total plus one SHAP fit.

**How to read this notebook.** Every code cell has a plain-language *"How to read
this"* note.

## 1. Load data and rebuild the CV setup

Same feature table and the same patient-grouped CV helper as notebook 03. We fix
the baseline model as **raw CatBoost** (the configuration you chose).

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)

data = pd.read_parquet(os.path.join("data", "features_track1.parquet"))
BOOKKEEPING = ["id", "subject_id", "split", "y"]
all_features = [c for c in data.columns if c not in BOOKKEEPING]

y = data["y"].astype(int).values
groups = data["subject_id"].values
CATEGORICAL = [c for c in ["gender", "ethnicity"] if c in all_features]

# decision thresholds from the seed-stability run
BASELINE_AUROC = 0.8032
SEED_STD       = 0.0011
HOLD_FLOOR     = BASELINE_AUROC - 0.002    # cut "holds" if AUROC >= this
IMPROVE_CEIL   = BASELINE_AUROC + 0.0022   # real improvement if AUROC >= this
print(f"baseline {BASELINE_AUROC:.4f} | hold floor {HOLD_FLOOR:.4f} | improve ceiling {IMPROVE_CEIL:.4f}")
print("total features:", len(all_features))

baseline 0.8032 | hold floor 0.8012 | improve ceiling 0.8054
total features: 247


**How to read this:** we loaded the table and set the numeric thresholds that
decide whether each cut is acceptable. **Remember:** these thresholds come from
your seed run — they're what make "did it hold?" a real test rather than a guess.

## 2. One evaluation helper (fixed folds)

We fix a single 5-fold patient-grouped split (seed 42) and reuse it for every
ablation, so all cuts are compared on identical folds. The helper trains raw
CatBoost on a given feature subset and returns mean AUROC / PR-AUC.

In [2]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(data[all_features], y, groups))

def make_model():
    return CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                              random_seed=42, verbose=False)

def eval_subset(feature_list):
    """Mean AUROC/PR-AUC of raw CatBoost on the given features, over fixed folds."""
    Xsub = data[feature_list]
    cats = [c for c in CATEGORICAL if c in feature_list]
    aucs, prs = [], []
    for tr, va in folds:
        m = make_model()
        m.fit(Xsub.iloc[tr], y[tr], cat_features=cats)
        p = m.predict_proba(Xsub.iloc[va])[:, 1]
        aucs.append(roc_auc_score(y[va], p))
        prs.append(average_precision_score(y[va], p))
    return np.mean(aucs), np.mean(prs)

def verdict(auroc):
    if auroc >= IMPROVE_CEIL: return "IMPROVED"
    if auroc >= HOLD_FLOOR:   return "held"
    return "DROPPED"

# confirm the full-set baseline on THESE folds (sanity: should be ~0.803)
full_auc, full_pr = eval_subset(all_features)
print(f"full set ({len(all_features)} feats): AUROC {full_auc:.4f}  PR-AUC {full_pr:.4f}  -> {verdict(full_auc)}")

full set (247 feats): AUROC 0.8025  PR-AUC 0.6308  -> held


**How to read this:** `eval_subset` is the workhorse — give it a list of
features, get back the CV score. The full-set number should land near 0.803,
confirming these folds match your earlier work. **Remember:** every ablation
below calls this same helper, so all comparisons are apples-to-apples.

## 3. Name the feature groups

We label the by-construction blocks so we can drop them by name. This is the
"redundancy filter" — but targeted at the exact overlapping groups we built,
rather than a blind correlation cut.

In [3]:
def cols_with(prefix):
    return [c for c in all_features if c.startswith(prefix)]

grp = {
    "lab_last":      cols_with("lab_last__"),
    "lab_flag":      cols_with("lab_flag__"),
    "lab_frac":      cols_with("lab_frac__"),
    "med_flag":      cols_with("med_flag__"),
    "med_intensity": cols_with("med_intensity__"),
    "icd_flag":      cols_with("icd_flag__"),
}
for k, v in grp.items():
    print(f"{k:14s}: {len(v)} features")
print("\nsingletons kept always: age, gender, ethnicity, sequence_length, icd_count, med_count")

lab_last      : 29 features
lab_flag      : 19 features
lab_frac      : 35 features
med_flag      : 40 features
med_intensity : 40 features
icd_flag      : 78 features

singletons kept always: age, gender, ethnicity, sequence_length, icd_count, med_count


**How to read this:** each group is a block of related features. The counts
should match the design (36 per lab view, 41 per med view, 91 icd flags).
**Remember:** the small "singleton" features (age, counts, sequence_length) are
cheap and known-useful from the EDA, so we never ablate those.

## 4. Tier 1 — structural redundancy cuts (4 runs)

Each cut tests a specific by-construction overlap. We start from the full set and
remove one group at a time, checking against the hold floor. The biggest
potential cut is the 91 ICD flags — the EDA showed ICD is sparse and no ICD flag
reached the top importances, so `icd_count` alone may capture its signal.

In [4]:
tier1_tests = {
    "drop lab_frac (keep last+flag)":      grp["lab_frac"],
    "drop lab_flag (keep last only)":      grp["lab_flag"],
    "drop med_intensity (keep flag+count)":grp["med_intensity"],
    "drop icd_flag (keep icd_count only)": grp["icd_flag"],
}

tier1_results = []
for name, drop_cols in tier1_tests.items():
    kept = [c for c in all_features if c not in drop_cols]
    auc, pr = eval_subset(kept)
    tier1_results.append((name, len(drop_cols), auc, pr, verdict(auc)))
    print(f"{name:38s} -{len(drop_cols):3d} feats | AUROC {auc:.4f} | {verdict(auc)}")

pd.DataFrame(tier1_results, columns=["cut", "n_dropped", "auroc", "prauc", "verdict"])

drop lab_frac (keep last+flag)         - 35 feats | AUROC 0.7893 | DROPPED
drop lab_flag (keep last only)         - 19 feats | AUROC 0.8008 | DROPPED
drop med_intensity (keep flag+count)   - 40 feats | AUROC 0.7974 | DROPPED
drop icd_flag (keep icd_count only)    - 78 feats | AUROC 0.8013 | held


,cut,n_dropped,auroc,prauc,verdict
0,drop lab_frac (keep last+flag),35,0.789296,0.592303,DROPPED
1,drop lab_flag (keep last only),19,0.800769,0.630024,DROPPED
2,drop med_intensity (keep flag+count),40,0.797355,0.627131,DROPPED
3,drop icd_flag (keep icd_count only),78,0.801349,0.628585,held


**How to read this:** each row drops one group and re-scores. "held" = safe to
cut (same AUROC, fewer features); "DROPPED" = the group mattered, keep it;
"IMPROVED" = cutting actually helped. **Remember:** these are tested
independently against the full set — we combine the winners in the next cell,
because dropping two groups together isn't guaranteed to behave like dropping
each alone.

### 4b. Combine the Tier-1 cuts that held

We now remove *all* the groups that held (or improved) together, and confirm the
combined smaller set still holds. This is the working set going into Tier 2.

In [5]:
# EDIT THIS LIST based on the Tier-1 table above: include every group whose
# verdict was "held" or "IMPROVED".
groups_to_drop = ["lab_frac", "icd_flag"]   # <-- example; adjust to your results

drop_cols = [c for g in groups_to_drop for c in grp[g]]
working_set = [c for c in all_features if c not in drop_cols]

auc, pr = eval_subset(working_set)
print(f"combined drop {groups_to_drop}: -{len(drop_cols)} feats "
      f"-> {len(working_set)} left | AUROC {auc:.4f} | {verdict(auc)}")

combined drop ['lab_frac', 'icd_flag']: -113 feats -> 134 left | AUROC 0.7917 | DROPPED


**How to read this:** edit `groups_to_drop` to match which Tier-1 cuts held,
then this confirms the combined set. If the combined verdict is still "held" or
"IMPROVED", `working_set` is your new, smaller feature list. **Remember:** if
combining unexpectedly "DROPPED", one group was compensating for another — back
off the more aggressive cut and re-check.

## 5. Tier 2 — SHAP-ranked tail cuts (1 SHAP fit + 3 runs)

Now we rank the *remaining* features by SHAP importance and try trimming the
low-ranked tail. SHAP is computed once, on a sample of rows (enough for a stable
ranking, much faster than all rows).

In [6]:
# fit once on a train-like sample and compute SHAP on a subset of rows for speed
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(data), size=min(3000, len(data)), replace=False)

cats_ws = [c for c in CATEGORICAL if c in working_set]
shap_model = make_model()
shap_model.fit(data[working_set].iloc[sample_idx], y[sample_idx], cat_features=cats_ws)

pool = Pool(data[working_set].iloc[sample_idx], y[sample_idx], cat_features=cats_ws)
shap_vals = shap_model.get_feature_importance(type="ShapValues", data=pool)
mean_abs = np.abs(shap_vals[:, :-1]).mean(0)          # last col is base value
shap_rank = pd.Series(mean_abs, index=working_set).sort_values(ascending=False)

print("SHAP top 15 (cross-check against EDA: expect age, med_count, seq_length, labs):")
print(shap_rank.head(15).round(4))
print("\nSHAP bottom 10 (candidates to cut):")
print(shap_rank.tail(10).round(4))

SHAP top 15 (cross-check against EDA: expect age, med_count, seq_length, labs):
age                                       0.2470
lab_last__Hematocrit Blood                0.2459
med_intensity__ELECT/CALORIC/H2O          0.2332
sequence_length                           0.2220
med_intensity__ANTIBIOTICS                0.1774
lab_last__Hemoglobin Blood                0.1600
med_count                                 0.1354
gender                                    0.1047
med_intensity__PSYCHOTHERAPEUTIC DRUGS    0.1022
med_intensity__ANTIPLATELET DRUGS         0.0957
med_intensity__SEDATIVE/HYPNOTICS         0.0932
med_flag__SEDATIVE/HYPNOTICS              0.0927
lab_last__Calcium, Total Blood            0.0925
lab_last__pH Blood                        0.0827
med_intensity__GASTROINTESTINAL           0.0743
dtype: float64

SHAP bottom 10 (candidates to cut):
lab_last__Eosinophils Ascites    0.0
lab_last__Basophils Blood        0.0
lab_flag__Lactate Blood          0.0
lab_flag__pO2 Blood   

**How to read this:** the top of this list should look like the EDA —
age, `med_count`, `sequence_length`, `icd_count`, common blood labs. If it does,
trust the ranking. **Remember:** SHAP *ranks* but doesn't tell us where to cut —
the next cell tests actual cutoffs. A feature ranked low isn't proven useless
until an ablation says so.

In [7]:
ranked = shap_rank.index.tolist()
n = len(ranked)
tier2_tests = {
    "keep top 50%":        ranked[: n // 2],
    "keep top 75%":        ranked[: (3 * n) // 4],
    "keep top 30 only":    ranked[:30],
}

tier2_results = []
for name, keep in tier2_tests.items():
    auc, pr = eval_subset(keep)
    tier2_results.append((name, len(keep), auc, pr, verdict(auc)))
    print(f"{name:18s} {len(keep):3d} feats | AUROC {auc:.4f} | {verdict(auc)}")

pd.DataFrame(tier2_results, columns=["cut", "n_kept", "auroc", "prauc", "verdict"])

keep top 50%        67 feats | AUROC 0.7916 | DROPPED
keep top 75%       100 feats | AUROC 0.7921 | DROPPED
keep top 30 only    30 feats | AUROC 0.7754 | DROPPED


,cut,n_kept,auroc,prauc,verdict
0,keep top 50%,67,0.791567,0.593492,DROPPED
1,keep top 75%,100,0.792117,0.594358,DROPPED
2,keep top 30 only,30,0.775425,0.575005,DROPPED


**How to read this:** three cutoffs down the SHAP ranking. Find the *smallest*
`n_kept` whose verdict is still "held" — that's your elbow, the point where
further cutting starts to hurt. **Remember:** if even "keep top 30" holds, you
have a dramatically smaller model at the same performance; if "keep top 50%"
already DROPPED, the tail was carrying signal and you keep more.

## 6. Tier 3 — confirm the final set across seeds

Pick the smallest feature set that held, and verify it matches the full model
across several CV seeds — the same stability check we ran on the full model, so
the comparison is fair.

In [8]:
# EDIT: set this to the smallest feature list that held (from Tier 1 combine or Tier 2)
FINAL_FEATURES = working_set   # <-- replace with e.g. tier2 "keep top 30" list if that held

SEEDS = [0, 1, 2, 42, 2024]
seed_aucs = []
for s in SEEDS:
    cv_s = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=s)
    folds_s = list(cv_s.split(data[FINAL_FEATURES], y, groups))
    cats_f = [c for c in CATEGORICAL if c in FINAL_FEATURES]
    aucs = []
    for tr, va in folds_s:
        m = make_model()
        m.fit(data[FINAL_FEATURES].iloc[tr], y[tr], cat_features=cats_f)
        aucs.append(roc_auc_score(y[va], m.predict_proba(data[FINAL_FEATURES].iloc[va])[:, 1]))
    seed_aucs.append(np.mean(aucs))
    print(f"seed {s:>4}: AUROC {np.mean(aucs):.4f}")

seed_aucs = np.array(seed_aucs)
print(f"\nFINAL set: {len(FINAL_FEATURES)} features (was {len(all_features)})")
print(f"AUROC across seeds: mean {seed_aucs.mean():.4f}  std {seed_aucs.std():.4f}"
      f"  range [{seed_aucs.min():.4f}, {seed_aucs.max():.4f}]")
print(f"full-model baseline was: mean 0.8032")

seed    0: AUROC 0.7900
seed    1: AUROC 0.7898
seed    2: AUROC 0.7859
seed   42: AUROC 0.7917
seed 2024: AUROC 0.7895

FINAL set: 134 features (was 247)
AUROC across seeds: mean 0.7894  std 0.0019  range [0.7859, 0.7917]
full-model baseline was: mean 0.8032


**How to read this:** compare the reduced model's multi-seed mean to the full
model's 0.8032. Within ~0.002 = you got the same performance with fewer features
(a clear win). Above 0.8054 = a genuine improvement. **Remember:** report the
mean across seeds, not the best seed — the point of this step is an honest number
for the smaller model.

## 7. Save the selected feature list

In [9]:
import json
with open(os.path.join("data", "selected_features.json"), "w") as f:
    json.dump(FINAL_FEATURES, f, indent=2)
print("saved", len(FINAL_FEATURES), "features to data/selected_features.json")

saved 134 features to data/selected_features.json


**How to read this:** the chosen feature list is saved for the final model.
**What's next:** retrain the chosen model on this reduced set for the final
submission, and/or move to Track 2 (the lab-sequence model) if you want to push
AUROC beyond this plateau — feature *selection* trims and clarifies, but new
*representations* are what break a plateau.

**Summary to record:** full set was ~270 features at AUROC 0.8032; record your
final feature count and its multi-seed AUROC here, with the verdict
(held / improved).